# Baseline Experiment: TF-IDF + Logistic Regression

First compare the original and balanced models. Then try 12 settings to choose the best baseline.

Use train for learning and dev for choosing settings.

## 1. Load the Prepared Train and Dev Data


In [1]:
# load the processed train and dev data
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"

train_df = pd.read_csv(DATA_DIR / "humanitarian_train.tsv", sep="\t")
dev_df = pd.read_csv(DATA_DIR / "humanitarian_dev.tsv", sep="\t")

print("Train:", len(train_df))
print("Dev:", len(dev_df))

Train: 61164
Dev: 8934


## 2. Initial Model Comparison

### 2.1 Train the Original Model


In [2]:
# build and train the baseline model
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

baseline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42,
    )),
])

# text_clean is the column with the preprocessed text
# class_label is the target variable
baseline.fit(train_df["text_clean"], train_df["class_label"])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](11,)","['affected_individual','caution_and_advice','displaced_and_evacuations', ...,'requests_or_needs','response_efforts','sympathy_and_support']"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


### 2.2 Evaluate the Original Model on Dev


In [3]:
# evaulate the model on the dev set 
from sklearn.metrics import classification_report, f1_score

dev_predictions = baseline.predict(dev_df["text_clean"])

print("Dev macro-F1:",
      round(f1_score(
          dev_df["class_label"],
          dev_predictions,
          average="macro",
      ), 4))

print(classification_report(
    dev_df["class_label"],
    dev_predictions,
    digits=3,
    zero_division=0,
))

Dev macro-F1: 0.6038
                                     precision    recall  f1-score   support

                affected_individual      0.846     0.645     0.732       366
                 caution_and_advice      0.705     0.537     0.610       307
          displaced_and_evacuations      0.667     0.226     0.338        53
          donation_and_volunteering      0.740     0.742     0.741       761
infrastructure_and_utilities_damage      0.717     0.582     0.642       509
             injured_or_dead_people      0.817     0.693     0.749       270
           missing_and_found_people      0.542     0.236     0.329        55
                   not_humanitarian      0.853     0.956     0.902      5259
                  requests_or_needs      0.872     0.853     0.862       702
                   response_efforts      0.211     0.035     0.061       113
               sympathy_and_support      0.818     0.575     0.675       539

                           accuracy                  

### 2.3 Train and Evaluate the Balanced Model

In [4]:
balanced_baseline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    )),
])

balanced_baseline.fit(
    train_df["text_clean"],
    train_df["class_label"],
)

balanced_predictions = balanced_baseline.predict(dev_df["text_clean"])

print(classification_report(
    dev_df["class_label"],
    balanced_predictions,
    digits=3,
    zero_division=0,
))

                                     precision    recall  f1-score   support

                affected_individual      0.711     0.740     0.726       366
                 caution_and_advice      0.494     0.743     0.593       307
          displaced_and_evacuations      0.312     0.566     0.403        53
          donation_and_volunteering      0.684     0.761     0.720       761
infrastructure_and_utilities_damage      0.556     0.684     0.613       509
             injured_or_dead_people      0.703     0.833     0.763       270
           missing_and_found_people      0.327     0.618     0.428        55
                   not_humanitarian      0.960     0.768     0.853      5259
                  requests_or_needs      0.776     0.913     0.839       702
                   response_efforts      0.165     0.540     0.253       113
               sympathy_and_support      0.577     0.746     0.650       539

                           accuracy                          0.767      89

### 2.4 Compare the Two Initial Models

In [5]:
comparison = pd.DataFrame({
    "model": ["Original", "Balanced"],
    "dev_macro_f1": [
        f1_score(dev_df["class_label"], dev_predictions, average="macro"),
        f1_score(dev_df["class_label"], balanced_predictions, average="macro"),
    ],
})

comparison

,model,dev_macro_f1
0,Original,0.603776
1,Balanced,0.621835


### 2.5 Inspect Errors from the Better Initial Model


In [6]:
# inspect mistakes for the 2nd model since it has the higher macro-F1 score
best_predictions = (
    balanced_predictions
    if float(comparison.at[1, "dev_macro_f1"])
    > float(comparison.at[0, "dev_macro_f1"])
    else dev_predictions
)

errors = dev_df.copy()
errors["predicted_label"] = best_predictions
errors = errors[
    errors["class_label"] != errors["predicted_label"]
]

display(errors[
    ["text_clean", "class_label", "predicted_label"]
].sample(min(10, len(errors)), random_state=42))

,text_clean,class_label,predicted_label
2425,God help us! hagupit,not_humanitarian,requests_or_needs
6374,Didn't feel any aftershocks last night. NepalE...,not_humanitarian,response_efforts
5752,RT USER : PHOTO via USER : Children are all ou...,affected_individual,donation_and_volunteering
6121,God Bless Nepal�__F�_�B,not_humanitarian,response_efforts
3156,Just saw a c-117 on approach for yyc. CAF yycf...,affected_individual,not_humanitarian
1572,It seems the whole Santiago is in dark. A mass...,requests_or_needs,infrastructure_and_utilities_damage
8027,"From Monte Carlo, insurersâ€™ view of the impa...",not_humanitarian,infrastructure_and_utilities_damage
1323,The earthquake destroyed our homes in Gonaives...,requests_or_needs,infrastructure_and_utilities_damage
6715,RT USER : PrayForThePhilippines Philippines Ma...,sympathy_and_support,caution_and_advice
6048,"Keep safe and be prepared. God is with us, Let...",not_humanitarian,sympathy_and_support


### 2.6 Initial Results

| Dev metric | Original | Balanced |
|---|---:|---:|
| Macro-F1 | 0.6038 | **0.6218** |
| Accuracy | **83.0%** | 76.7% |
| `requests_or_needs` recall | 85.3% | **91.3%** |
| `response_efforts` recall | 3.5% | **54.0%** |

The balanced model catches more tweets from smaller categories, but also makes more incorrect predictions. For example, only 16.5% of its `response_efforts` predictions are correct.

The balanced model wins this first comparison because it has higher macro-F1. Next, we try more settings before choosing the final baseline. We save the test set for final evaluation.

## 3. Compare Settings Using the Dev Set

Try 12 combinations of:

- single words or single words plus two-word phrases
- `C` values of 0.1, 1, or 10, which control how strongly the model is regularized
- ordinary or balanced class weights

Choose the model with the highest dev macro-F1, which gives all 11 categories equal importance

The model reads `text_clean` and predicts `class_label`.
It learns from train and uses dev to compare settings.
The test set is saved for final evaluation.

The shared functions are in `src/baseline.py`.
This section can run without running the earlier cells

### 3.1 Set Up the Shared Experiment Functions


In [7]:
from pathlib import Path
import sys
import json
import hashlib
import platform
from importlib.metadata import version
import joblib
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src/baseline.py").is_file():
    ROOT = ROOT.parent
assert (ROOT / "src/baseline.py").is_file(), "Run from root or notebooks/"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.baseline import load_split, run_experiments, dev_metrics, per_class_report, save_pipeline

DATA_DIR = ROOT / "data/processed"
train_df = load_split(DATA_DIR / "humanitarian_train.tsv")
dev_df = load_split(DATA_DIR / "humanitarian_dev.tsv")
print(f"Train: {len(train_df):,}; dev: {len(dev_df):,}; labels: {train_df.class_label.nunique()}")


Train: 61,164; dev: 8,934; labels: 11


### 3.2 Run and Compare the 12 Settings


In [8]:
# fit TF-IDF only on training text
# each classifier is also trained only on train
# the function caches the two training feature matrices to avoid repeated work
experiment_results, selected_model = run_experiments(train_df, dev_df)
print(experiment_results.to_string(index=False))


 1/12 ngrams=(1, 1), C=0.1, weights=None: dev macro-F1=0.4206
 2/12 ngrams=(1, 1), C=0.1, weights=balanced: dev macro-F1=0.5544
 3/12 ngrams=(1, 1), C=1.0, weights=None: dev macro-F1=0.6038
 4/12 ngrams=(1, 1), C=1.0, weights=balanced: dev macro-F1=0.6218
 5/12 ngrams=(1, 1), C=10.0, weights=None: dev macro-F1=0.6310
 6/12 ngrams=(1, 1), C=10.0, weights=balanced: dev macro-F1=0.6286
 7/12 ngrams=(1, 2), C=0.1, weights=None: dev macro-F1=0.3514
 8/12 ngrams=(1, 2), C=0.1, weights=balanced: dev macro-F1=0.5282
 9/12 ngrams=(1, 2), C=1.0, weights=None: dev macro-F1=0.5686
10/12 ngrams=(1, 2), C=1.0, weights=balanced: dev macro-F1=0.6080
11/12 ngrams=(1, 2), C=10.0, weights=None: dev macro-F1=0.6346
12/12 ngrams=(1, 2), C=10.0, weights=balanced: dev macro-F1=0.6407
ngram_range    C class_weight  dev_macro_f1  dev_weighted_f1  dev_accuracy  iterations
     (1, 2) 10.0     balanced      0.640742         0.811599      0.804343         150
     (1, 2) 10.0         none      0.634594         0.

## 4. Selected Model: Dev Results and Errors

Choose the model with the highest dev macro-F1

Check how well it predicts each category and where it makes mistakes

These results help us choose the baseline model

In [9]:
selected_predictions = selected_model.predict(dev_df["text_clean"])
selected_scores = dev_metrics(dev_df["class_label"], selected_predictions)
selected_settings = {
    "ngram_range": list(selected_model.named_steps["tfidf"].ngram_range),
    "C": selected_model.named_steps["classifier"].C,
    "class_weight": selected_model.named_steps["classifier"].class_weight,
}
selected_report = per_class_report(selected_model, dev_df)
print("Selected settings:", selected_settings)
print("Dev metrics:", selected_scores)
print(selected_report.to_string())

# full texts are saved to help with inspecting errors without truncation
dev_output = dev_df[["id", "event", "text_clean", "class_label"]].copy()
dev_output["predicted_label"] = selected_predictions
dev_output["correct"] = dev_output["class_label"].eq(dev_output["predicted_label"])
print("Incorrect dev predictions:", int((~dev_output["correct"]).sum()))


Selected settings: {'ngram_range': [1, 2], 'C': 10.0, 'class_weight': 'balanced'}
Dev metrics: {'macro_f1': 0.640741592307217, 'weighted_f1': 0.8115990019470872, 'accuracy': 0.8043429594806357}
                                     precision    recall  f1-score      support
affected_individual                   0.752137  0.721311  0.736402   366.000000
caution_and_advice                    0.562341  0.719870  0.631429   307.000000
displaced_and_evacuations             0.381818  0.396226  0.388889    53.000000
donation_and_volunteering             0.635714  0.818660  0.715681   761.000000
infrastructure_and_utilities_damage   0.621107  0.705305  0.660534   509.000000
injured_or_dead_people                0.715719  0.792593  0.752197   270.000000
missing_and_found_people              0.459016  0.509091  0.482759    55.000000
not_humanitarian                      0.947585  0.831907  0.885986  5259.000000
requests_or_needs                     0.791563  0.908832  0.846154   702.000000
respon

## 5. Save the Model and Dev Results

Save the trained model, experiment scores, dev predictions, and settings

Check that loading the saved model gives the same predictions

The trained model is saved at `checkpoints/baseline/selected_pipeline.joblib`

In [10]:
TABLES = ROOT / "results/tables"
TABLES.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ROOT / "checkpoints/baseline/selected_pipeline.joblib"
experiment_results.to_csv(TABLES / "baseline_dev_experiments.csv", index=False)
selected_report.to_csv(TABLES / "baseline_dev_classification_report.csv", index_label="label")
dev_output.to_csv(TABLES / "baseline_dev_predictions.tsv", sep="\t", index=False)
dev_output.loc[~dev_output["correct"]].to_csv(
    TABLES / "baseline_dev_errors.tsv", sep="\t", index=False
)
save_pipeline(selected_model, MODEL_PATH)
reloaded = joblib.load(MODEL_PATH)
assert (reloaded.predict(dev_df["text_clean"]) == selected_predictions).all()
assert len(experiment_results) == 12
assert abs(experiment_results.iloc[0]["dev_macro_f1"] - selected_scores["macro_f1"]) < 1e-12

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

metadata = {
    "input_column": "text_clean", "target_column": "class_label",
    "selected_settings": selected_settings,
    "selection_metric": "dev_macro_f1", "dev_metrics": selected_scores,
    "labels": selected_model.classes_.tolist(),
    "training_rows": len(train_df), "dev_rows": len(dev_df),
    "trained_on": "train only; dev used only for selection",
    "test_evaluated": False, "random_state": 42,
    "max_iter": 1000, "solver": selected_model.named_steps["classifier"].solver,
    "tfidf_other_settings": "scikit-learn defaults for recorded version",
    "python": platform.python_version(),
    "packages": {name: version(name) for name in ["scikit-learn", "pandas", "numpy", "scipy", "joblib", "threadpoolctl"]},
    "data_sha256": {split: sha256(DATA_DIR / f"humanitarian_{split}.tsv") for split in ["train", "dev"]},
    "model_path": str(MODEL_PATH.relative_to(ROOT)),
    "model_sha256": sha256(MODEL_PATH),
}
(TABLES / "baseline_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
print("Saved results to results/tables/ and pipeline to", MODEL_PATH.relative_to(ROOT))
print("Passed: 12 experiments, selected score matches, saved model reproduces dev predictions.")


Saved results to results/tables/ and pipeline to checkpoints/baseline/selected_pipeline.joblib
Passed: 12 experiments, selected score matches, saved model reproduces dev predictions.


## 6. Baseline Findings

The selected model uses **unigrams + bigrams**, **C = 10**, and **balanced class weights**.

| Development result | Value |
|---|---:|
| Selected macro-F1 | 0.6407 |
| Selected weighted-F1 | 0.8116 |
| Selected accuracy | 80.43% |
| Initial ordinary model macro-F1 | 0.6038 |
| Initial balanced model macro-F1 | 0.6218 |

The selected model improves macro-F1 by **3.70 percentage points** over the initial ordinary model. The unweighted bigram model has higher accuracy (83.88%) but lower macro-F1 (0.6346). We select the balanced model because macro-F1 is our main selection metric.

`requests_or_needs` recall is 90.88%, with precision of 79.16%. `response_efforts` remains difficult, with an F1 of 0.2537 and precision of 21.94%.

All 12 fits converged. Reloading the saved pipeline produces the same dev predictions.

These are dev results used for model selection. Final test evaluation and cross-crisis experiments remain for the group's evaluation stage.

## Baseline Model Summary

- Loaded the cleaned training and dev datasets
- Trained TF-IDF + Logistic Regression
- Compared 12 settings using the dev set
- Selected single words and two-word phrases, `C=10`, and balanced class weights
- Achieved a dev macro-F1 of **0.6407** and accuracy of **80.43%**
- Saved the trained model, results, and incorrect predictions

The model learned from training data only. The dev set helped us choose settings. The test set has not been evaluated.